<a href="https://colab.research.google.com/github/MoLue/wft_digital_medicine/blob/main/agentic_ai_medical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Author: Timo Lüders*

*Level: Mixed (Beginner to Advanced) | Duration: ~45-60 minutes | Tags: `agentic-ai`, `agents`, `rag`, `multi-agent`, `safety`, `medical-kb`, `compact`*

# Agentic AI in Medicine

## What You Will Learn

This notebook introduces **AI agents** in a medical context. Unlike plain LLMs that only generate text, agents can:
- Use **tools** (call Python functions, look up databases)
- **Decide** which tool to use based on user input
- **Retrieve** relevant medical knowledge before answering (RAG)
- Work together as **multi-agent systems**

### Notebook Structure (pick your path!)

| Part | Level | Topic | Duration |
|------|-------|-------|----------|
| Part 1 | Beginner | Tool-Using Medical Agent | ~15 min |
| Part 2 | Intermediate | RAG-Enhanced Medical Agent | ~15 min |
| Part 3 | Advanced | Multi-Agent Routing | ~15 min |
| Part 4 | Advanced | Safety & Guardrails | ~10 min |
| Discussion | All | Ethics, Risks, EU AI Act | ~5 min |

**Beginners**: Focus on Parts 1-2. **Advanced**: Try all parts.

### Technical Notes
- **No API keys required** — we use open-source models only (BioGPT, BioBERT)
- Runs on **free Google Colab** (CPU or GPU)
- All medical data is **synthetic** — not for real patient care!

> **Disclaimer:** This notebook is for educational purposes only. It is **not** a medical device and must not be used for real patient care or decision-making.

## 0. Setup

First, we install the required packages and load our models. This may take 1-2 minutes on Colab.

In [ ]:
# Install required packages (run once per Colab session)
!pip install -q transformers torch psutil numpy sacremoses protobuf

print("Packages installed.")

In [ ]:
import sys
import os
import psutil
import torch
import torch.nn.functional as F
import numpy as np
from typing import List, Dict, Optional, Tuple

IN_COLAB = "google.colab" in sys.modules
has_gpu = torch.cuda.is_available()
mem_gb = psutil.virtual_memory().available / 1e9

print(f"Running in Colab: {IN_COLAB}")
print(f"GPU available: {has_gpu}")
print(f"Available memory: {mem_gb:.1f} GB")

if mem_gb < 2.0:
    print("WARNING: Low memory. BioGPT (~1.5 GB) may fail. Consider restarting the runtime.")

In [ ]:
from transformers import BioGptTokenizer, BioGptForCausalLM
from transformers import AutoTokenizer, AutoModel

# Load BioGPT for text generation (~1.5 GB download on first run)
print("Loading BioGPT (this may take a minute)...")
biogpt_tokenizer = BioGptTokenizer.from_pretrained("microsoft/biogpt")
biogpt_model = BioGptForCausalLM.from_pretrained(
    "microsoft/biogpt",
    device_map="auto" if has_gpu else None,
    dtype=torch.float16 if has_gpu else torch.float32,
)
print("BioGPT loaded.")

# Load BioBERT for embeddings (~400 MB)
print("Loading BioBERT...")
biobert_tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.2")
biobert_model = AutoModel.from_pretrained("dmis-lab/biobert-base-cased-v1.2")
if has_gpu:
    biobert_model = biobert_model.to("cuda")
print("BioBERT loaded.")

In [ ]:
# Helper functions used throughout the notebook


def generate_text(prompt: str, max_new_tokens: int = 150) -> str:
    """Generate text using BioGPT given a prompt."""
    formatted = f"Question: {prompt}\n\nAnswer:"
    inputs = biogpt_tokenizer(formatted, return_tensors="pt")
    if has_gpu:
        inputs = {k: v.to(biogpt_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = biogpt_model.generate(
            inputs["input_ids"],
            max_new_tokens=max_new_tokens,
            temperature=0.8,
            top_p=0.92,
            top_k=50,
            repetition_penalty=1.2,
            pad_token_id=biogpt_tokenizer.eos_token_id,
            attention_mask=inputs.get("attention_mask"),
        )

    full_text = biogpt_tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "Answer:" in full_text:
        return full_text.split("Answer:", 1)[1].strip()
    return full_text.strip()


def get_embedding(text: str) -> torch.Tensor:
    """Get a BioBERT sentence embedding (CLS token) for the given text."""
    inputs = biobert_tokenizer(
        text, return_tensors="pt", padding=True, truncation=True, max_length=512
    )
    if has_gpu:
        inputs = {k: v.to(biobert_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = biobert_model(**inputs)

    return outputs.last_hidden_state[:, 0, :].squeeze(0).cpu()


def cosine_sim(a: torch.Tensor, b: torch.Tensor) -> float:
    """Compute cosine similarity between two 1D tensors."""
    return F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()


print("Helper functions ready.")

---

## Part 1: Tool-Using Medical Agent (Beginner)

### What is an AI Agent?

A **plain LLM** receives text and produces text. An **agent** extends this by:

```
User Question
     ↓
Agent decides: which tool should I call?
     ↓
Tool executes (Python function) → returns result
     ↓
Agent uses result + LLM to generate user-facing answer
```

In this part, we build a medical agent with three simple **tools**:
1. **BMI Calculator** — computes Body Mass Index
2. **Blood Pressure Classifier** — classifies BP readings
3. **Drug Interaction Checker** — looks up known interactions from a dictionary

### 1.1 Define the Tools

Each tool is a Python function. The agent will choose which one to call based on the user's question.

In [ ]:
def tool_bmi_calculator(weight_kg: float, height_m: float) -> Dict:
    """Calculate BMI and return the value with WHO classification."""
    bmi = weight_kg / (height_m ** 2)
    if bmi < 18.5:
        category = "Underweight"
    elif bmi < 25.0:
        category = "Normal weight"
    elif bmi < 30.0:
        category = "Overweight"
    else:
        category = "Obese"
    return {"bmi": round(bmi, 1), "category": category}


def tool_bp_classifier(systolic: int, diastolic: int) -> Dict:
    """Classify blood pressure according to AHA guidelines."""
    if systolic < 120 and diastolic < 80:
        category = "Normal"
    elif systolic < 130 and diastolic < 80:
        category = "Elevated"
    elif systolic < 140 or diastolic < 90:
        category = "High Blood Pressure Stage 1"
    elif systolic >= 140 or diastolic >= 90:
        category = "High Blood Pressure Stage 2"
    else:
        category = "Unknown"

    if systolic > 180 or diastolic > 120:
        category = "Hypertensive Crisis - Seek emergency care!"

    return {"systolic": systolic, "diastolic": diastolic, "category": category}


# Simple drug interaction database (educational, NOT for real use)
DRUG_INTERACTIONS = {
    ("metformin", "alcohol"): "Metformin with alcohol increases risk of lactic acidosis.",
    ("warfarin", "aspirin"): "Concurrent use increases bleeding risk significantly.",
    ("lisinopril", "potassium"): "ACE inhibitors with potassium supplements may cause hyperkalemia.",
    ("metformin", "contrast dye"): "Metformin should be paused before contrast imaging (risk of lactic acidosis).",
    ("simvastatin", "grapefruit"): "Grapefruit inhibits CYP3A4, increasing simvastatin levels and toxicity risk.",
    ("ssri", "tramadol"): "Combined serotonergic drugs increase risk of serotonin syndrome.",
}


def tool_drug_interaction(drug_a: str, drug_b: str) -> Dict:
    """Check for known interactions between two drugs/substances."""
    key1 = (drug_a.lower().strip(), drug_b.lower().strip())
    key2 = (drug_b.lower().strip(), drug_a.lower().strip())

    if key1 in DRUG_INTERACTIONS:
        return {"interaction_found": True, "details": DRUG_INTERACTIONS[key1]}
    elif key2 in DRUG_INTERACTIONS:
        return {"interaction_found": True, "details": DRUG_INTERACTIONS[key2]}
    else:
        return {
            "interaction_found": False,
            "details": f"No known interaction found between '{drug_a}' and '{drug_b}' in our database.",
        }


# Quick test
print("BMI:", tool_bmi_calculator(85, 1.78))
print("BP:", tool_bp_classifier(145, 92))
print("Drug:", tool_drug_interaction("warfarin", "aspirin"))

### 1.2 Build the Agent

The agent uses simple keyword matching to decide which tool to call, then passes the tool result to BioGPT for a natural-language response.

In [ ]:
import re


AGENT_SYSTEM_PROMPT = """You are a medical information assistant. You help with:
- BMI calculations and interpretation
- Blood pressure classification
- Drug interaction checks

IMPORTANT: You provide general health information only.
You are NOT a doctor. Always recommend consulting a healthcare professional.
""".strip()


def detect_tool(user_message: str) -> str:
    """Decide which tool to call based on keyword matching."""
    text = user_message.lower()
    if any(kw in text for kw in ["bmi", "body mass", "weight", "height", "overweight", "obese"]):
        return "bmi"
    if any(kw in text for kw in ["blood pressure", "bp", "systolic", "diastolic", "hypertension"]):
        return "bp"
    if any(kw in text for kw in ["interaction", "drug", "medication", "combine", "mix", "together with"]):
        return "drug_interaction"
    return "general"


def extract_numbers(text: str) -> List[float]:
    """Extract all numbers from a text string."""
    return [float(x) for x in re.findall(r"\d+\.?\d*", text)]


def run_medical_agent(user_message: str) -> str:
    """Process a user message through the medical agent."""
    tool_name = detect_tool(user_message)
    tool_result = None
    numbers = extract_numbers(user_message)

    if tool_name == "bmi" and len(numbers) >= 2:
        weight, height = numbers[0], numbers[1]
        if height > 3:  # likely in cm, convert to m
            height = height / 100
        tool_result = tool_bmi_calculator(weight, height)

    elif tool_name == "bp" and len(numbers) >= 2:
        tool_result = tool_bp_classifier(int(numbers[0]), int(numbers[1]))

    elif tool_name == "drug_interaction":
        # Try to extract drug names (simplified: look for quoted or capitalized words)
        words = user_message.lower().split()
        drug_candidates = [
            w for w in words
            if w in ["metformin", "warfarin", "aspirin", "lisinopril", "potassium",
                     "simvastatin", "grapefruit", "alcohol", "ssri", "tramadol",
                     "contrast", "dye"]
        ]
        if "contrast" in drug_candidates and "dye" in drug_candidates:
            drug_candidates = [d for d in drug_candidates if d != "dye"]
            drug_candidates = ["contrast dye" if d == "contrast" else d for d in drug_candidates]
        if len(drug_candidates) >= 2:
            tool_result = tool_drug_interaction(drug_candidates[0], drug_candidates[1])

    # Build the prompt for BioGPT
    context = ""
    if tool_result:
        context = f"Tool result: {tool_result}"
    else:
        context = "No tool was called (insufficient data or unrecognized query)."

    prompt = f"""{AGENT_SYSTEM_PROMPT}

User question: \"{user_message}\"
{context}

Provide a helpful, concise answer. Always recommend consulting a healthcare professional."""

    return generate_text(prompt, max_new_tokens=200)


print("Medical agent ready!")

In [ ]:
# Test the agent with different queries
test_queries = [
    "I weigh 92 kg and I am 1.75 m tall. What is my BMI?",
    "My blood pressure reading is 155/95. Is that normal?",
    "Can I take warfarin and aspirin together?",
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"User: {query}")
    print(f"Agent: {run_medical_agent(query)}")

### ✅ Exercise 1: Add a New Tool

Create a new tool function `tool_heart_rate_zone(age: int, resting_hr: int)` that:
1. Calculates the maximum heart rate (simple formula: 220 - age)
2. Returns heart rate zones (e.g., fat burn: 50-70% of max, cardio: 70-85%, peak: 85-100%)

Then:
- Add keywords to `detect_tool` so the agent can recognize heart rate questions
- Add a branch in `run_medical_agent` that calls your new tool
- Test it with: "I am 35 years old with a resting heart rate of 72 bpm"

In [ ]:
# Your code here:


---

## Part 2: RAG-Enhanced Medical Agent (Intermediate)

### What is RAG?

**Retrieval-Augmented Generation (RAG)** lets the agent look up relevant information from a knowledge base before generating an answer. Think of it as an "open-book exam" for the AI.

```
User Question
     ↓
Agent embeds the question (BioBERT)
     ↓
Search knowledge base for most relevant passages
     ↓
Pass retrieved context + question to BioGPT
     ↓
Generate grounded answer
```

We use 5 medical condition files as our knowledge base.

### 2.1 Build the Knowledge Base

We load medical condition texts and split them into chunks. Each chunk gets a BioBERT embedding for semantic search.

In [ ]:
# Medical knowledge base content
# (In a real system, these would be loaded from files or a database)

MEDICAL_KB = {
    "diabetes": """Diabetes mellitus is a group of metabolic disorders characterized by a high blood sugar level over a prolonged period of time. Symptoms often include frequent urination, increased thirst and increased appetite. If left untreated, diabetes can cause many health complications.

Type 1 diabetes results from the pancreas's failure to produce enough insulin due to loss of beta cells. This form was previously referred to as "insulin-dependent diabetes mellitus" or "juvenile diabetes".

Type 2 diabetes begins with insulin resistance, a condition in which cells fail to respond to insulin properly. The most common cause is a combination of excessive body weight and insufficient exercise.

Prevention and treatment involve maintaining a healthy diet, regular physical exercise, a normal body weight, and avoiding use of tobacco. Type 1 diabetes must be managed with insulin injections. Type 2 diabetes may be treated with medications with or without insulin.

Recent advances in diabetes care include closed-loop insulin delivery systems (artificial pancreas technology), continuous glucose monitoring (CGM) with sensors offering up to 14 days wear time, and novel medications like GLP-1 receptor agonists (semaglutide, tirzepatide) and SGLT2 inhibitors (empagliflozin, dapagliflozin) which offer cardiovascular and renal benefits beyond glucose control.""",

    "hypertension": """Hypertension, also known as high blood pressure, is a long-term medical condition in which the blood pressure in the arteries is persistently elevated. High blood pressure typically does not cause symptoms but is a major risk factor for stroke, coronary artery disease, heart failure, and chronic kidney disease.

Blood pressure is expressed by two measurements: systolic and diastolic pressures. For most adults, high blood pressure is present if the resting blood pressure is persistently at or above 130/80 or 140/90 mmHg.

Lifestyle changes include weight loss, physical exercise, decreased salt intake, reducing alcohol intake, and a healthy diet. The DASH diet can lower systolic blood pressure by 8-14 mmHg.

First-line pharmacological treatments include thiazide diuretics, calcium channel blockers (CCBs), ACE inhibitors, and ARBs. Combination therapy is often required. Recent evidence supports initial combination therapy for patients with stage 2 hypertension.""",

    "asthma": """Asthma is a long-term inflammatory disease of the airways of the lungs. It is characterized by variable and recurring symptoms, reversible airflow obstruction, and easily triggered bronchospasms. Symptoms include episodes of wheezing, coughing, chest tightness, and shortness of breath.

Asthma is classified according to the frequency of symptoms, FEV1, and peak expiratory flow rate. Modern understanding recognizes it as a heterogeneous disease with multiple phenotypes including allergic asthma, non-allergic asthma, and late-onset asthma.

Treatment follows a stepwise approach. A significant recent change is the recommendation against SABA monotherapy even for mild asthma. All patients should receive ICS-containing treatment. For severe asthma, biologic therapies like omalizumab (anti-IgE), mepolizumab (anti-IL-5), and dupilumab (anti-IL-4R) have revolutionized treatment.""",

    "coronary_artery_disease": """Coronary artery disease (CAD) involves the reduction of blood flow to the heart muscle due to atherosclerosis. It is the most common cardiovascular disease. Types include stable angina, unstable angina, myocardial infarction, and sudden cardiac death.

Risk factors include high blood pressure, smoking, diabetes, lack of exercise, obesity, high blood cholesterol, and poor diet. Modern diagnostics include coronary CT angiography (CCTA) with CT-FFR for functional assessment.

Treatment includes lifestyle changes, medications (antiplatelets, statins, beta-blockers, ACE inhibitors, PCSK9 inhibitors), and procedures like PCI with drug-eluting stents or CABG surgery. SGLT2 inhibitors and GLP-1 receptor agonists have shown cardiovascular benefits regardless of diabetes status.""",

    "alzheimers": """Alzheimer's disease (AD) is a neurodegenerative disease causing 60-70% of dementia cases. The most common early symptom is difficulty remembering recent events. The disease process involves amyloid plaques, neurofibrillary tangles, and loss of neuronal connections.

Biomarkers for early diagnosis include CSF markers (decreased amyloid-beta-42, increased tau), PET scans for amyloid/tau, and emerging blood-based biomarkers. The FDA approved aducanumab (2021) and lecanemab (2023) as disease-modifying therapies targeting amyloid, though these remain controversial.

Current symptomatic treatments include cholinesterase inhibitors (donepezil, rivastigmine, galantamine) and memantine. AI and machine learning are enabling earlier diagnosis through neuroimaging and digital biomarker analysis.""",
}

print(f"Knowledge base loaded: {len(MEDICAL_KB)} conditions")
for name, text in MEDICAL_KB.items():
    print(f"  - {name}: {len(text)} characters")

In [ ]:
# Split knowledge base into chunks and compute embeddings


def chunk_text(text: str, chunk_size: int = 300) -> List[str]:
    """Split text into chunks of approximately chunk_size characters at paragraph boundaries."""
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks = []
    current_chunk = ""

    for para in paragraphs:
        if len(current_chunk) + len(para) < chunk_size:
            current_chunk += ("\n\n" + para if current_chunk else para)
        else:
            if current_chunk:
                chunks.append(current_chunk)
            current_chunk = para

    if current_chunk:
        chunks.append(current_chunk)

    return chunks


# Build the indexed knowledge base
kb_chunks: List[Dict] = []

print("Building knowledge base index (computing embeddings)...")
for condition, text in MEDICAL_KB.items():
    chunks = chunk_text(text)
    for i, chunk in enumerate(chunks):
        embedding = get_embedding(chunk)
        kb_chunks.append({
            "condition": condition,
            "chunk_id": i,
            "text": chunk,
            "embedding": embedding,
        })

print(f"Indexed {len(kb_chunks)} chunks from {len(MEDICAL_KB)} conditions.")

### 2.2 Semantic Search (Retrieval)

Given a question, we find the most relevant chunks by comparing BioBERT embeddings.

In [ ]:
def retrieve_relevant_chunks(query: str, top_k: int = 3) -> List[Dict]:
    """Find the top_k most relevant knowledge base chunks for a query."""
    query_emb = get_embedding(query)

    scored = []
    for chunk in kb_chunks:
        score = cosine_sim(query_emb, chunk["embedding"])
        scored.append({**chunk, "score": score})

    scored.sort(key=lambda x: x["score"], reverse=True)
    return scored[:top_k]


# Test retrieval
test_question = "What are the latest treatments for type 2 diabetes?"
results = retrieve_relevant_chunks(test_question)

print(f"Query: {test_question}\n")
for r in results:
    print(f"  [{r['condition']}] score={r['score']:.3f}")
    print(f"  {r['text'][:150]}...\n")

### 2.3 RAG Agent

Combine retrieval with generation: the agent first retrieves relevant knowledge, then passes it as context to BioGPT.

In [ ]:
def run_rag_agent(user_question: str) -> str:
    """Answer a medical question using RAG (retrieval + generation)."""
    # Step 1: Retrieve relevant knowledge
    chunks = retrieve_relevant_chunks(user_question, top_k=2)
    context = "\n\n".join([c["text"] for c in chunks])
    sources = ", ".join(set([c["condition"] for c in chunks]))

    # Step 2: Generate answer with context
    prompt = f"""You are a medical information assistant with access to a knowledge base.
Use the following medical knowledge to answer the question accurately.
If the knowledge base does not contain enough information, say so.
Always recommend consulting a healthcare professional.

Knowledge base context:
{context}

User question: \"{user_question}\"

Provide a concise, accurate answer based on the context above."""

    answer = generate_text(prompt, max_new_tokens=250)

    return f"{answer}\n\n[Sources: {sources}]"


# Test the RAG agent
rag_questions = [
    "What are the recent biologic therapies for severe asthma?",
    "How does the DASH diet help with hypertension?",
    "What biomarkers are used for early Alzheimer's detection?",
]

for q in rag_questions:
    print(f"\n{'='*60}")
    print(f"User: {q}")
    print(f"Agent: {run_rag_agent(q)}")

### ✅ Exercise 2: Extend the Knowledge Base

Add a new condition to the knowledge base:

1. Choose a condition you're interested in (e.g., depression, COPD, stroke)
2. Write 3-4 paragraphs about it (symptoms, treatment, recent advances)
3. Add it to `MEDICAL_KB` and re-run the indexing cell
4. Test the RAG agent with a question about your new condition

Does the agent find and use your new content?

In [ ]:
# Your code here:


---

## Part 3: Multi-Agent Routing (Advanced)

In real-world systems, different types of questions need different specialists. We build a **router** that directs queries to the right agent:

```
User Query
     ↓
Router (BioBERT cosine similarity)
     ↓
     ├── Medical Info Agent (RAG)
     ├── Tool Agent (BMI, BP, drugs)
     └── Admin Agent (appointments, general)
```

In [ ]:
# Define agent descriptions for the router

AGENT_REGISTRY = {
    "medical_info": {
        "description": "Questions about medical conditions, diseases, symptoms, treatments, medications, and recent medical research.",
        "handler": run_rag_agent,
    },
    "tool_agent": {
        "description": "Calculations and lookups: BMI calculation, blood pressure classification, drug interaction checks, and health metric computations.",
        "handler": run_medical_agent,
    },
    "admin": {
        "description": "Administrative questions: appointment scheduling, clinic opening hours, general non-medical inquiries, and organizational matters.",
        "handler": lambda q: generate_text(
            f"You are a clinic receptionist. Answer this administrative question concisely: {q}",
            max_new_tokens=100,
        ),
    },
}

# Pre-compute embeddings for agent descriptions
agent_embeddings = {
    name: get_embedding(info["description"])
    for name, info in AGENT_REGISTRY.items()
}

print("Agent registry ready with", len(AGENT_REGISTRY), "agents.")

In [ ]:
def route_to_agent(user_message: str) -> Tuple[str, float]:
    """Route a user message to the most appropriate agent using BioBERT similarity."""
    query_emb = get_embedding(user_message)

    best_agent = None
    best_score = -1.0

    for agent_name, agent_emb in agent_embeddings.items():
        score = cosine_sim(query_emb, agent_emb)
        if score > best_score:
            best_score = score
            best_agent = agent_name

    return best_agent, best_score


def run_multi_agent_system(user_message: str) -> str:
    """Route a message to the appropriate agent and return the response."""
    agent_name, score = route_to_agent(user_message)
    handler = AGENT_REGISTRY[agent_name]["handler"]

    print(f"  [Router] Directed to: {agent_name} (confidence: {score:.3f})")

    return handler(user_message)


# Test the multi-agent system
multi_agent_tests = [
    "What are the biologic therapies for asthma?",
    "I weigh 80 kg and am 170 cm tall. What is my BMI?",
    "When is the clinic open on Saturdays?",
    "Can I take warfarin and aspirin at the same time?",
    "What are the early signs of Alzheimer's disease?",
]

for msg in multi_agent_tests:
    print(f"\n{'='*60}")
    print(f"User: {msg}")
    response = run_multi_agent_system(msg)
    print(f"Agent: {response}")

### ✅ Exercise 3: Add a Patient Education Agent

Create a new agent called `patient_education` that:
1. Uses the RAG system to retrieve medical information
2. Rewrites it in **simple, patient-friendly language** (no jargon, short sentences)
3. Add it to `AGENT_REGISTRY` with an appropriate description
4. Re-compute agent embeddings

Test with: "Can you explain diabetes to me in simple terms?"

Hint: Write a handler function that calls `retrieve_relevant_chunks`, then uses a prompt that asks BioGPT to explain in layperson language.

In [ ]:
# Your code here:


---

## Part 4: Safety & Guardrails (Advanced)

Medical AI systems **must** have safety boundaries. In this section, we add guardrails that:
1. Detect when a question is about **emergency symptoms** and urge the user to call emergency services
2. Detect when the agent is being asked for a **definitive diagnosis** and refuse
3. Add a **confidence threshold** — if the router is unsure, ask for clarification

In [ ]:
EMERGENCY_KEYWORDS = [
    "chest pain", "can't breathe", "cannot breathe", "heart attack",
    "stroke", "unconscious", "severe bleeding", "anaphylaxis",
    "suicidal", "suicide", "overdose",
]

DIAGNOSIS_KEYWORDS = [
    "do i have", "am i sick", "diagnose me", "is this cancer",
    "what disease do i have", "tell me my diagnosis",
]

CONFIDENCE_THRESHOLD = 0.75


def safety_check(user_message: str) -> Optional[str]:
    """Check for safety-critical patterns. Returns a safety response or None."""
    text = user_message.lower()

    # Emergency detection
    for kw in EMERGENCY_KEYWORDS:
        if kw in text:
            return (
                "EMERGENCY DETECTED: If you or someone else is in immediate danger, "
                "please call emergency services (112 in Europe, 911 in the US) immediately. "
                "This AI system cannot provide emergency medical assistance."
            )

    # Diagnosis request detection
    for kw in DIAGNOSIS_KEYWORDS:
        if kw in text:
            return (
                "I cannot provide medical diagnoses. I can share general health information, "
                "but for a diagnosis, please consult a qualified healthcare professional. "
                "Would you like general information about a specific condition instead?"
            )

    return None


def run_safe_multi_agent(user_message: str) -> str:
    """Multi-agent system with safety guardrails."""
    # Step 1: Safety check
    safety_response = safety_check(user_message)
    if safety_response:
        return f"[SAFETY] {safety_response}"

    # Step 2: Route with confidence check
    agent_name, score = route_to_agent(user_message)

    if score < CONFIDENCE_THRESHOLD:
        return (
            f"[LOW CONFIDENCE: {score:.2f}] I'm not sure I understand your question correctly. "
            f"Could you rephrase it? I can help with:\n"
            f"  - Medical information (conditions, treatments)\n"
            f"  - Health calculations (BMI, blood pressure)\n"
            f"  - Administrative questions (appointments, hours)"
        )

    # Step 3: Call the agent
    print(f"  [Router] {agent_name} (confidence: {score:.3f})")
    handler = AGENT_REGISTRY[agent_name]["handler"]
    return handler(user_message)


# Test safety system
safety_tests = [
    "I'm having severe chest pain and difficulty breathing",
    "Do I have diabetes? My blood sugar was 200.",
    "What are the risk factors for coronary artery disease?",
    "My blood pressure is 130/85, is that okay?",
    "asdkjhfkajshf random gibberish",
]

for msg in safety_tests:
    print(f"\n{'='*60}")
    print(f"User: {msg}")
    print(f"Response: {run_safe_multi_agent(msg)}")

### ✅ Exercise 4: Improve the Safety Filter

1. Add at least **3 more emergency keywords** and **3 more diagnosis keywords**
2. Create a test set of 10 messages (mix of safe, emergency, and diagnosis requests)
3. For each message, check if the safety filter gives the correct response
4. Identify any **false positives** (safe message blocked) or **false negatives** (dangerous message not caught)

Write a brief reflection (2-3 sentences) in a Markdown cell about the limitations of keyword-based safety filters.

In [ ]:
# Your code here:


---

## Discussion: Ethics, Risks, and the Future

### Key Questions to Reflect On

1. **How does this differ from ChatGPT?**
   - Our agents use open-source models (BioGPT, BioBERT) running locally
   - No data leaves your computer/Colab instance
   - Much smaller and less capable than GPT-4, but transparent and controllable
   - Custom tools and safety filters give us explicit control over behavior

2. **What are the risks?**
   - **Hallucinations**: BioGPT can generate plausible-sounding but incorrect medical information
   - **Incomplete safety filters**: Keyword-based filters miss many dangerous queries
   - **Bias**: Models trained on English biomedical literature may not represent all populations
   - **Over-reliance**: Users may trust AI output without verification

3. **EU AI Act implications**
   - Medical AI systems are classified as **high-risk** under the EU AI Act
   - Requirements: risk management, data governance, transparency, human oversight
   - Our educational system would need significant hardening for any real deployment

4. **What would it take to deploy this in a real clinic?**
   - Comprehensive testing and validation against medical standards
   - Integration with electronic health records (EHR)
   - Regulatory approval (CE marking for medical devices in the EU)
   - Continuous monitoring and human-in-the-loop oversight
   - Data privacy compliance (GDPR, patient consent)

### Further Reading
- [EU AI Act Summary](https://artificialintelligenceact.eu/)
- [WHO guidance on AI for health](https://www.who.int/publications/i/item/9789240029200)
- The full admin-focused agents notebook: `llm_agents_healthcare_agents.ipynb`
- The multi-agent Streamlit demo: `streamlit_multi_agents_diabetes/`

## What's Next?

| Notebook | Level | What You'll Learn |
|----------|-------|-------------------|
| [LLM Agents for Healthcare Admin](https://colab.research.google.com/github/MoLue/wft_digital_medicine/blob/main/llm_agents_healthcare_agents.ipynb) | Advanced | Build agents for appointments, billing, multi-agent routing |
| [LLMs in Healthcare](https://colab.research.google.com/github/MoLue/wft_digital_medicine/blob/main/llms_in_healthcare.ipynb) | Advanced | Prompt engineering, RAG, model evaluation |
| [NLP & Transformers](https://colab.research.google.com/github/MoLue/wft_digital_medicine/blob/main/dm_nlp.ipynb) | Intermediate | Classical NLP pipeline, medical NER |
| [Heart Disease Prediction](https://colab.research.google.com/github/MoLue/wft_digital_medicine/blob/main/heart_disease_prediction_analysis.ipynb) | Intermediate | ML classification on real cardiovascular data |